In [0]:
from pyspark.sql import SparkSession
import random

spark = SparkSession.builder.getOrCreate()

random.seed(42)

categories = {
    "Electronics": ["Laptop","Mobile","Tablet"],
    "Fashion": ["Shirt","Jeans","Jacket"],
    "Home": ["Chair","Table","Sofa"],
    "Sports": ["Football","Cricket Bat","Tennis Racket"]
}

brands = ["Samsung","Apple","Sony","Nike","Adidas"]

products = []

for i in range(1,501):

    category = random.choice(list(categories.keys()))
    subcategory = random.choice(categories[category])
    products.append((i,f"{subcategory}_{i}",category,subcategory,random.choice(brands),
                 round(random.uniform(10,1000),2)))

df = spark.createDataFrame(products,["ProductID","ProductName","Category","SubCategory","Brand","CostPrice"])
df.show(10)

print("Total Records =", df.count())

+---------+-----------+-----------+-----------+-------+---------+
|ProductID|ProductName|   Category|SubCategory|  Brand|CostPrice|
+---------+-----------+-----------+-----------+-------+---------+
|        1|   Laptop_1|Electronics|     Laptop|   Sony|   252.44|
|        2|   Jacket_2|    Fashion|     Jacket|Samsung|   679.93|
|        3|   Tablet_3|Electronics|     Tablet|   Nike|    41.46|
|        4|   Laptop_4|Electronics|     Laptop|  Apple|    510.3|
|        5|   Tablet_5|Electronics|     Tablet|  Apple|   718.86|
|        6| Football_6|     Sports|   Football|   Nike|   593.37|
|        7|   Laptop_7|Electronics|     Laptop|   Nike|   346.85|
|        8|    Shirt_8|    Fashion|      Shirt|   Sony|   111.19|
|        9| Football_9|     Sports|   Football|   Sony|   849.02|
|       10|   Chair_10|       Home|      Chair|   Nike|   540.87|
+---------+-----------+-----------+-----------+-------+---------+
only showing top 10 rows
Total Records = 500


In [0]:
%sql
SHOW SCHEMAS IN retailanalytics;

databaseName
audit
bronze
config
default
gold
history
information_schema
reject
silver
watermark


In [0]:
%sql
CREATE VOLUME IF NOT EXISTS retailanalytics.default.retail_volume;

In [0]:
%sql
SHOW VOLUMES IN retailanalytics.default;

database,volume_name
default,retail_volume


In [0]:
product_path = "/Volumes/retailanalytics/default/retail_volume/landing/products"

(df.write.mode("overwrite").option("header", True).csv(product_path))
print("Products saved successfully")

Products saved successfully


In [0]:
product_path = "/Volumes/retailanalytics/default/retail_volume/landing/products"

(df.coalesce(1).write.mode("overwrite").option("header", True).csv(product_path))
print("Products saved successfully")

Products saved successfully


In [0]:
files = dbutils.fs.ls(product_path)

part_file = [f.path for f in files if f.name.startswith("part-")][0]
dbutils.fs.cp(part_file,f"{product_path}/products.csv")

print("products.csv created")

products.csv created


In [0]:
display(dbutils.fs.ls("/Volumes/retailanalytics/default/retail_volume/landing/products"))

path,name,size,modificationTime
dbfs:/Volumes/retailanalytics/default/retail_volume/landing/products/_SUCCESS,_SUCCESS,0,1782365135000
dbfs:/Volumes/retailanalytics/default/retail_volume/landing/products/_committed_2680218658790780587,_committed_2680218658790780587,850,1782365135000
dbfs:/Volumes/retailanalytics/default/retail_volume/landing/products/_committed_4485157773377863319,_committed_4485157773377863319,113,1782365101000
dbfs:/Volumes/retailanalytics/default/retail_volume/landing/products/_started_2680218658790780587,_started_2680218658790780587,0,1782365134000
dbfs:/Volumes/retailanalytics/default/retail_volume/landing/products/_started_4485157773377863319,_started_4485157773377863319,0,1782365101000
dbfs:/Volumes/retailanalytics/default/retail_volume/landing/products/part-00000-tid-2680218658790780587-19481e3d-d1f9-4e0f-8a25-2450226a5eb0-146-1-c000.csv,part-00000-tid-2680218658790780587-19481e3d-d1f9-4e0f-8a25-2450226a5eb0-146-1-c000.csv,2661,1782365134000
dbfs:/Volumes/retailanalytics/default/retail_volume/landing/products/part-00001-tid-2680218658790780587-19481e3d-d1f9-4e0f-8a25-2450226a5eb0-147-1-c000.csv,part-00001-tid-2680218658790780587-19481e3d-d1f9-4e0f-8a25-2450226a5eb0-147-1-c000.csv,2685,1782365134000
dbfs:/Volumes/retailanalytics/default/retail_volume/landing/products/part-00002-tid-2680218658790780587-19481e3d-d1f9-4e0f-8a25-2450226a5eb0-142-1-c000.csv,part-00002-tid-2680218658790780587-19481e3d-d1f9-4e0f-8a25-2450226a5eb0-142-1-c000.csv,2877,1782365135000
dbfs:/Volumes/retailanalytics/default/retail_volume/landing/products/part-00003-tid-2680218658790780587-19481e3d-d1f9-4e0f-8a25-2450226a5eb0-143-1-c000.csv,part-00003-tid-2680218658790780587-19481e3d-d1f9-4e0f-8a25-2450226a5eb0-143-1-c000.csv,2851,1782365135000
dbfs:/Volumes/retailanalytics/default/retail_volume/landing/products/part-00004-tid-2680218658790780587-19481e3d-d1f9-4e0f-8a25-2450226a5eb0-144-1-c000.csv,part-00004-tid-2680218658790780587-19481e3d-d1f9-4e0f-8a25-2450226a5eb0-144-1-c000.csv,2789,1782365134000
